# Lab 2: Ask for the analysis, then check it
*Python for Healthcare · Stanford Technology Training (ITS-2519) · Dr. Lionel Levine*
**Session 1 · 45 minutes: about 35 minutes of work, then we debrief together**

You'll ask Gemini for the trial's primary analysis twice: once with a vague request, then with the analysis plan.
Then you'll check its work with six checks and audit a draft that contains planted problems.

**The data.** `analysis_s1.csv` has one clean row per randomized participant. It's the table you built in Lab 1.

## Step 0. Load the table

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/lionellevine/python-for-healthcare/main/data/session1/"

def load(name):
    """Read a course file: your uploaded copy if there is one, otherwise the course repo."""
    return pd.read_csv(name) if Path(name).exists() else pd.read_csv(DATA_URL + name)

from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

def fmt_p(p):
    """Format a p-value the way papers do."""
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

trial = load("analysis_s1.csv")
print(trial.shape)
trial.head()

## Step 1. Read the analysis plan (3 minutes)
**Home BP Coaching Trial: statistical analysis plan (excerpt)**
- **Population:** analyze everyone by the arm they were randomized to. The primary model uses everyone with a
  week-12 SBP; nobody is excluded for a missing baseline characteristic. (A full plan would add a sensitivity
  analysis for the missing outcomes.)
- **Primary outcome:** office systolic BP at week 12 (mmHg).
- **Effect:** coaching minus usual care. Usual care is the reference group.
- **Primary model:** linear regression of week-12 SBP on arm, adjusted for baseline SBP and site
  (the randomization strata).
- **Report:** the effect in mmHg with a 95% confidence interval and p-value, and n per arm.
- **Secondary:** the unadjusted difference, with Welch's t-test.
- **Table 1:** baseline characteristics by arm, with missing counts.
- **Figure:** mean SBP at week 0 and week 12 by arm, with 95% confidence intervals.

## Step 2. The vague request (5 minutes)
Open the Gemini panel and type exactly: `Analyze this trial and tell me whether coaching worked.`
Let it write and run its code. Don't correct it. Then fill in the table: what did it decide on its own?

| Decision | What the plan says | What Gemini chose |
|---|---|---|
| Primary outcome | Week-12 SBP |  |
| Who's included | Everyone with a week-12 SBP |  |
| Adjusted for | Baseline SBP and site |  |
| Model or test | Linear regression, plus Welch's t-test |  |
| Direction of the effect | Coaching minus usual care |  |
| Anything extra |  |  |

Every blank in your request is a decision the assistant makes for you. Some of its choices may be reasonable.
They still aren't your plan's choices.

## Step 3. Give Gemini the plan (8 minutes)
First, run this cell for a fresh copy of the table, in case the vague request changed `trial`.

In [ ]:
trial = load("analysis_s1.csv")   # a fresh copy
print(trial.shape)

1. In the Gemini panel, open **Custom Instructions**, paste the text in the box below, and save it to this notebook.
   (Can't find Custom Instructions? Paste the text as your first message instead.)
2. Send these two requests, one at a time. After each one, insert and run the code Gemini writes.
   - `Build Table 1.`
   - `Run the primary analysis.`

The unadjusted comparison and the figure are written for you below them.

```text
You are my coding assistant in a Google Colab notebook. A pandas DataFrame named `trial` is loaded (one row per randomized participant in a synthetic trial). Columns: participant_id, site (North/South), arm (coaching/usual care), randomization_date, age, sex (F/M), bmi, diabetes (yes/no), smoker (yes/no), n_bp_meds, sbp_w0, dbp_w0, sbp_w12, dbp_w12 (mmHg; w0 = baseline, w12 = week 12).
Follow this analysis plan exactly:
1. Population: analyze everyone by the arm they were randomized to. Do not drop rows for missing values, except that outcome models use the rows that have sbp_w12. Never call dropna() on the whole table.
2. Effect = coaching minus usual care, with usual care as the reference group.
3. Primary model: ordinary least squares, sbp_w12 ~ arm + sbp_w0 + site (statsmodels formula), with usual care as the reference level of arm. Report the coaching coefficient, its 95% CI, the p-value, and n per arm.
4. Unadjusted comparison: Welch's t-test on sbp_w12 (equal_var=False), with the mean difference and its 95% CI.
5. Table 1 by arm, with missing counts for every variable.
6. Print the number of rows used at every step.
Use pandas, statsmodels, scipy, and matplotlib. Explain each step in one sentence.
```

In [ ]:
# Table 1: insert and run the code Gemini writes for "Build Table 1."

In [ ]:
#@title Fallback: Table 1 (only if Gemini isn't available: tick the box, then run this cell)
run_fallback = False  #@param {type:"boolean"}
if run_fallback:
    def mean_sd(s):
        return f"{s.mean():.1f} ({s.std():.1f})"

    def n_pct(mask, s):
        return f"{mask.sum()} ({100 * mask.sum() / s.notna().sum():.1f}%)"

    columns = {}
    for arm, g in trial.groupby("arm"):
        columns[f"{arm} (n={len(g)})"] = {
            "Age, mean (SD)": mean_sd(g["age"]),
            "Female, n (%)": n_pct(g["sex"] == "F", g["sex"]),
            "BMI, mean (SD)": mean_sd(g["bmi"]),
            "   BMI missing, n": str(g["bmi"].isna().sum()),
            "Diabetes, n (%)": n_pct(g["diabetes"] == "yes", g["diabetes"]),
            "Current smoker, n (% of known)": n_pct(g["smoker"] == "yes", g["smoker"]),
            "   Smoking status missing, n": str(g["smoker"].isna().sum()),
            "BP medications, mean (SD)": mean_sd(g["n_bp_meds"]),
            "Baseline SBP, mean (SD)": mean_sd(g["sbp_w0"]),
            "Baseline DBP, mean (SD)": mean_sd(g["dbp_w0"]),
            "South clinic, n (%)": n_pct(g["site"] == "South", g["site"]),
        }
    print("Rows used for Table 1:", len(trial))
    display(pd.DataFrame(columns))

In [ ]:
# Primary analysis: insert and run the code Gemini writes for "Run the primary analysis."

In [ ]:
#@title Fallback: Primary analysis (only if Gemini isn't available: tick the box, then run this cell)
run_fallback = False  #@param {type:"boolean"}
if run_fallback:
    outcome = trial[trial["sbp_w12"].notna()]
    n = outcome["arm"].value_counts()
    print(f"Rows used: {len(outcome)} (coaching {n['coaching']}, usual care {n['usual care']})")

    model = smf.ols("sbp_w12 ~ C(arm, Treatment(reference='usual care')) + sbp_w0 + site", data=outcome).fit()
    term = "C(arm, Treatment(reference='usual care'))[T.coaching]"
    low, high = model.conf_int().loc[term]
    print(f"Coaching minus usual care: {model.params[term]:.2f} mmHg (95% CI {low:.2f} to {high:.2f}), "
          f"{fmt_p(model.pvalues[term])}, n = {int(model.nobs)}")
    print(f"R-squared: {model.rsquared:.2f}")

In [ ]:
# Provided: the unadjusted comparison (Welch's t-test)
outcome = trial[trial["sbp_w12"].notna()]
coach = outcome.loc[outcome["arm"] == "coaching", "sbp_w12"]
usual = outcome.loc[outcome["arm"] == "usual care", "sbp_w12"]
res = stats.ttest_ind(coach, usual, equal_var=False)
ci = res.confidence_interval()
print(f"Rows used: {len(coach) + len(usual)} (coaching {len(coach)}, usual care {len(usual)})")
print(f"Means: coaching {coach.mean():.1f}, usual care {usual.mean():.1f} mmHg")
print(f"Difference: {coach.mean() - usual.mean():.2f} mmHg (95% CI {ci.low:.2f} to {ci.high:.2f}), {fmt_p(res.pvalue)}")

In [ ]:
# Provided: the figure in the plan
fig, ax = plt.subplots(figsize=(6, 4))
for arm, marker, shift in [("coaching", "o", -0.2), ("usual care", "s", 0.2)]:
    g = trial[trial["arm"] == arm]
    means, half_widths = [], []
    for col in ["sbp_w0", "sbp_w12"]:
        x = g[col].dropna()
        means.append(x.mean())
        half_widths.append(stats.t.ppf(0.975, len(x) - 1) * x.std() / np.sqrt(len(x)))
    ax.errorbar([0 + shift, 12 + shift], means, yerr=half_widths, marker=marker, capsize=4, label=arm)
ax.set_xticks([0, 12], ["Week 0", "Week 12"])
ax.set_ylabel("Systolic BP, mmHg (mean and 95% CI)")
ax.set_title("Mean systolic BP by arm")
ax.legend(frameon=False)
plt.show()

## Step 4. Check it: six checks (9 minutes)
Don't trust a result because it looks thorough. Each check below recomputes something straight from the data,
without the assistant's code. Compare it with what Gemini reported. Start from a fresh copy:

In [ ]:
trial = load("analysis_s1.csv")   # a fresh copy, so the checks see the original data

**Check 1. The counts.** PASS if Table 1 shows everyone randomized in each arm, and every model reports the number
of people who have a week-12 SBP.

In [ ]:
print("Randomized, by arm (Table 1 should match):")
print(trial["arm"].value_counts(), "\n")
print("With a week-12 SBP, by arm (every model should use these):")
print(trial.loc[trial["sbp_w12"].notna(), "arm"].value_counts())

**Check 2. Nobody dropped for the wrong reason.** Search Gemini's code for `dropna()`. A `dropna()` without
`subset=` drops anyone with a blank anywhere, including participants whose smoking status was never asked.
PASS if every model uses all the rows with a week-12 SBP.

In [ ]:
print("Rows with a blank somewhere:", trial.isna().any(axis=1).sum())
print("Rows left after dropna() on the whole table:", len(trial.dropna()))
print("Rows with a week-12 SBP:", trial["sbp_w12"].notna().sum())

**Check 3. The direction.** The arm with the lower week-12 mean must come out lower in the model.
PASS if the effect is labeled *coaching minus usual care* and its sign agrees with these means.
Watch for a coefficient labeled `arm[T.usual care]`: that's usual care minus coaching.

In [ ]:
trial.groupby("arm")["sbp_w12"].mean().round(1)

**Check 4. The model matches the plan.** Read the formula Gemini used. PASS if it adjusts for baseline SBP
(`sbp_w0`) and `site`, and usual care is the reference group.

**Check 5. Effect, interval, and units.** PASS if the result gives the effect in mmHg with a 95% confidence
interval, not just a p-value.

**Check 6 (if you have time). The figure agrees with the numbers.** PASS if the figure's means match this table.

In [ ]:
(trial.melt(id_vars="arm", value_vars=["sbp_w0", "sbp_w12"], var_name="visit", value_name="sbp")
      .dropna(subset=["sbp"])
      .groupby(["arm", "visit"])["sbp"].agg(["count", "mean"])
      .round(1))

**Record the results.** Double-click to edit.

| Check | PASS or FAIL | Evidence |
|---|---|---|
| 1. Counts |  |  |
| 2. Nobody dropped for the wrong reason |  |  |
| 3. Direction |  |  |
| 4. Model matches the plan |  |  |
| 5. Effect, interval, units |  |  |
| 6. Figure agrees |  |  |

## Step 5. Audit a draft (7 minutes)
This draft was written for this lab and contains planted problems: the kind that survive a quick look.
Run it, then apply the six checks. Which checks fail, and why?

In [ ]:
# DRAFT ANALYSIS: written for this lab, with planted problems
d = load("analysis_s1.csv").dropna()
model = smf.ols("sbp_w12 ~ arm + sbp_w0 + site", data=d).fit()
effect = model.params["arm[T.usual care]"]
p = model.pvalues["arm[T.usual care]"]
print(f"Coaching changed week-12 SBP by {effect:+.1f} mmHg (p = {p:.4f}), n = {int(model.nobs)}")

t = stats.ttest_ind(d.loc[d["arm"] == "coaching", "sbp_w12"],
                    d.loc[d["arm"] == "usual care", "sbp_w12"])
print(f"t-test p = {t.pvalue:.4f}")

## Step 6. Write the result (3 minutes)
Write one sentence a reader could check: the comparison, the effect with its 95% CI, the adjustment, and n.

*Compared with usual care, coaching changed week-12 systolic BP by ___ mmHg (95% CI ___ to ___), adjusted for
baseline SBP and site (n = ___: ___ coaching, ___ usual care).*

## Before the debrief, jot down
1. Which check would you run first next time?
2. What did your vague request decide for you?
3. Write one thing this analysis must refuse to do.